## Tahap Persiapan : Inisialisasi SparkSession & Import Library


In [1]:
import time, json
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (SparkSession.builder
    .appName('TidarMart_2505060024')
    .master('local[2]')
    .config('spark.driver.memory', '1536m')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate())

HDFS_BASE = "hdfs://localhost:9000/tidarmart"
SEED = 60024
print("Spark UI:", spark.sparkContext.uiWebUrl)

Matplotlib is building the font cache; this may take a moment.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 03:34:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://zeroone:4040


## Tahap 1 : Penyimpanan Terdistribusi & Ingesti (HDFS)
• Buat struktur direktori di HDFS, misalnya /tidarmart/raw/transaksi, /tidarmart/raw/pelanggan,
/tidarmart/raw/produk, /tidarmart/stream_input, /tidarmart/curated, /tidarmart/models.

• Unggah seluruh data ke HDFS. Tampilkan bukti ukuran, jumlah blok, dan faktor replikasi (hdfs dfs -du -
h, hdfs fsck).

• Baca seluruh data dari HDFS (hdfs://localhost:9000/...) bukan dari path lokal pada tahap-tahap berikutnya.

• Jelaskan: mengapa data mentah dan data bersih dipisah direktori, dan apa arti replikasi = 1 pada VM
pseudo-distributed dibanding klaster nyata

In [2]:
# Membuat Direktori & Mengunggah Data ke HDFS
%%bash
hdfs dfs -mkdir -p /tidarmart/raw/transaksi /tidarmart/raw/pelanggan /tidarmart/raw/produk /tidarmart/raw/stream_input /tidarmart/curated /tidarmart/models
hdfs dfs -put -f data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
hdfs dfs -put -f data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
hdfs dfs -put -f data_proyek/produk.json /tidarmart/raw/produk/

In [4]:
# Verifikasi Ukuran, Blok, dan Replikasi (Bukti Volume)

!hdfs dfs -du -h /tidarmart/raw/
!hdfs fsck /tidarmart/raw/transaksi -files -blocks

6.0 M    6.0 M    /tidarmart/raw/pelanggan
685.3 K  685.3 K  /tidarmart/raw/produk
0        0        /tidarmart/raw/stream_input
187.2 M  187.2 M  /tidarmart/raw/transaksi
Connecting to namenode via http://localhost:9870/fsck?ugi=zerouno&files=1&blocks=1&path=%2Ftidarmart%2Fraw%2Ftransaksi
FSCK started by zerouno (auth:SIMPLE) from /127.0.0.1 for path /tidarmart/raw/transaksi at Thu Nov 26 03:42:41 UTC 2026

/tidarmart/raw/transaksi <dir>
/tidarmart/raw/transaksi/transaksi_part1.csv 49081412 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1327634461-10.0.2.15-1788789334784:blk_1073741866_1042 len=49081412 Live_repl=1

/tidarmart/raw/transaksi/transaksi_part2.csv 49081080 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1327634461-10.0.2.15-1788789334784:blk_1073741867_1043 len=49081080 Live_repl=1

/tidarmart/raw/transaksi/transaksi_part3.csv 49077294 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1327634461-10.0.2.15-1788789334784:blk_1073741868_1044 len=490772

## Tahap 2 : Profiling Kualitas Data (Veracity & Variety)

• Hitung jumlah baris, jumlah order_id unik, persentase nilai kosong per kolom, jumlah harga negatif,
distribusi qty (temukan outlier dengan kuantil, jenis nya bebas), dan variasi penulisan kota.

• Sajikan hasil profiling dalam satu tabel ringkasan “sebelum pembersihan”.

• Untuk profil pelanggan dan produk: tunjukkan bahwa atribut produk bervariasi menurut kategori (berapa
persen produk memiliki setiap atribut) dan bagaimana field tags diperlakukan.

In [5]:
# Membaca Data dari HDFS dengan Skema Eksplisit untuk JSON
df_trx_raw = spark.read.csv(f"{HDFS_BASE}/raw/transaksi/*.csv", header=True, inferSchema=True)

schema_pelanggan = StructType([
    StructField("pelanggan_id", IntegerType()), StructField("nama", StringType()),
    StructField("kota", StringType()), StructField("tanggal_daftar", StringType()),
    StructField("membership", StringType())
])
schema_produk = StructType([
    StructField("produk_id", IntegerType()), StructField("nama_produk", StringType()),
    StructField("kategori", StringType()), StructField("harga_dasar", LongType()),
    StructField("berat_kg", DoubleType()), StructField("garansi_bulan", IntegerType()),
    StructField("voltase", IntegerType()), StructField("ukuran", StringType()),
    StructField("bahan", StringType()), StructField("kadaluarsa_hari", IntegerType()),
    StructField("penerbit", StringType()), StructField("tags", ArrayType(StringType()))
])

df_pelanggan = spark.read.schema(schema_pelanggan).json(f"{HDFS_BASE}/raw/pelanggan/pelanggan.json")
df_produk = spark.read.schema(schema_produk).json(f"{HDFS_BASE}/raw/produk/produk.json")

In [6]:
# Profiling Transaksi
total_rows = df_trx_raw.count()
unique_orders = df_trx_raw.select("order_id").distinct().count()
neg_harga = df_trx_raw.filter(F.col("harga_satuan") < 0).count()
outlier_qty = df_trx_raw.filter(F.col("qty") > 50).count()
qty_q = df_trx_raw.approxQuantile("qty", [0.5, 0.99, 0.999, 1.0], 0.001)

null_qty = df_trx_raw.filter(F.col("qty").isNull() | F.isnan("qty")).count()
null_harga = df_trx_raw.filter(F.col("harga_satuan").isNull() | F.isnan("harga_satuan")).count()
variasi_kota = df_trx_raw.select("kota").distinct().count()

profil_sebelum = pd.DataFrame({
    "Metrik": ["Total Baris", "Order Unik", "Duplikat", "Null Qty (%)", "Null Harga (%)", "Harga Negatif", "Outlier Qty (>50)", "Variasi Nama Kota"],
    "Sebelum": [total_rows, unique_orders, total_rows - unique_orders, round(null_qty/total_rows*100, 3), round(null_harga/total_rows*100, 3), neg_harga, outlier_qty, variasi_kota]
})
print("Kuantil Qty [50%, 99%, 99.9%, 100%]:", qty_q)
display(profil_sebelum)

[Stage 24:=============================>                            (1 + 1) / 2]

Kuantil Qty [50%, 99%, 99.9%, 100%]: [4.0, 7.0, 999.0, 999.0]


,Metrik,Sebelum
0,Total Baris,2010000.000
1,Order Unik,2000000.000
2,Duplikat,10000.000
3,Null Qty (%),0.299
4,Null Harga (%),0.300
5,Harga Negatif,2005.000
6,Outlier Qty (>50),1035.000
7,Variasi Nama Kota,24.000


In [7]:
# Profiling Produk & Pelanggan
total_prod = df_produk.count()
print("Persentase Kelengkapan Atribut Produk (%):")
df_produk.select([
    F.round(F.count(F.col(c)) / total_prod * 100, 2).alias(c) for c in df_produk.columns
]).show()

# Cek penanganan field array opsional (tags)
df_produk.select("produk_id", "kategori", "tags").filter(F.col("tags").isNotNull()).show(5, truncate=False)

Persentase Kelengkapan Atribut Produk (%):
+---------+-----------+--------+-----------+--------+-------------+-------+------+-----+---------------+--------+-----+
|produk_id|nama_produk|kategori|harga_dasar|berat_kg|garansi_bulan|voltase|ukuran|bahan|kadaluarsa_hari|penerbit| tags|
+---------+-----------+--------+-----------+--------+-------------+-------+------+-----+---------------+--------+-----+
|    100.0|      100.0|   100.0|      100.0|   100.0|        12.52|  12.52| 13.68|13.68|          12.04|   11.82|30.52|
+---------+-----------+--------+-----------+--------+-------------+-------+------+-----+---------------+--------+-----+

+---------+---------+------------------------+
|produk_id|kategori |tags                    |
+---------+---------+------------------------+
|3        |Buku     |[terlaris, lokal, promo]|
|6        |Kesehatan|[promo]                 |
|11       |Fashion  |[promo, lokal]          |
|12       |Makanan  |[terlaris]              |
|13       |Olahraga |[lokal

## Tahap 3 : ETL & Penyimpanan Kolumnar

• Bersihkan transaksi sesuai temuan Tahap 2: deduplikasi, penanganan nilai kosong, harga negatif, outlier
qty, normalisasi nama kota. Setiap aturan harus disertai alasan (dibuang, diisi, atau dikoreksi, dan
mengapa).

• Tambahkan kolom turunan: total_nilai (qty × harga_satuan × (1 − diskon/100)), bulan,
hari_dalam_minggu, flag_batal.

• Simpan hasil ke HDFS /tidarmart/curated dalam format Parquet dengan partitionBy yang Anda pilih dan
justifikasikan (misalnya kota atau bulan).

• Bandingkan CSV dan Parquet: ukuran di HDFS dan waktu eksekusi satu query agregasi yang sama.
Sajikan dalam tabel.

• Tampilkan laporan “sesudah pembersihan” berdampingan dengan laporan sebelumnya.

In [9]:
# Pembersihan Data & Penambahan Kolom Turunan
df_clean = (df_trx_raw
    .dropDuplicates(["order_id"])                              
    .dropna(subset=["qty", "harga_satuan"])                    
    .withColumn("harga_satuan", F.abs(F.col("harga_satuan")))  
    .filter((F.col("qty") > 0) & (F.col("qty") <= 50))         
    .withColumn("kota", F.initcap(F.trim(F.col("kota"))))     
    .withColumn("tanggal_ts", F.to_timestamp("tanggal"))
    .withColumn("total_nilai", F.col("qty") * F.col("harga_satuan") * (1 - F.col("diskon_persen") / 100.0))
    .withColumn("bulan", F.date_format("tanggal_ts", "yyyy-MM"))
    .withColumn("hari_dalam_minggu", F.dayofweek("tanggal_ts"))
    .withColumn("flag_batal", F.when(F.col("status_pesanan") == "Dibatalkan", 1).otherwise(0))
)

In [10]:
# Menyimpan ke HDFS Format Parquet (Partisi "bulan")
df_clean.write.mode("overwrite").partitionBy("bulan").parquet(f"{HDFS_BASE}/curated/transaksi_parquet")
!hdfs dfs -du -h /tidarmart/curated/transaksi_parquet

0       0       /tidarmart/curated/transaksi_parquet/_SUCCESS
12.6 M  37.7 M  /tidarmart/curated/transaksi_parquet/bulan=2026-01
11.4 M  34.1 M  /tidarmart/curated/transaksi_parquet/bulan=2026-02
12.6 M  37.8 M  /tidarmart/curated/transaksi_parquet/bulan=2026-03
12.2 M  36.5 M  /tidarmart/curated/transaksi_parquet/bulan=2026-04
12.6 M  37.8 M  /tidarmart/curated/transaksi_parquet/bulan=2026-05
12.1 M  36.4 M  /tidarmart/curated/transaksi_parquet/bulan=2026-06


In [11]:
# Benchmark CSV vs Parquet & Tabel "Sesudah Pembersihan"
df_trx = spark.read.parquet(f"{HDFS_BASE}/curated/transaksi_parquet")

t0 = time.time(); df_trx_raw.groupBy("kota").agg(F.sum("qty")).collect(); t_csv = time.time() - t0
t0 = time.time(); df_trx.groupBy("kota").agg(F.sum("qty")).collect(); t_parq = time.time() - t0
print(f"Waktu Query -> CSV: {t_csv:.2f} dtk | Parquet: {t_parq:.2f} dtk")

# Perbandingan Sebelum vs Sesudah
tot_c = df_trx.count()
profil_sebelum["Sesudah"] = [
    tot_c, df_trx.select("order_id").distinct().count(), 0,
    df_trx.filter(F.col("qty").isNull()).count(), df_trx.filter(F.col("harga_satuan").isNull()).count(),
    df_trx.filter(F.col("harga_satuan") < 0).count(), df_trx.filter(F.col("qty") > 50).count(),
    df_trx.select("kota").distinct().count()
]
display(profil_sebelum)

Waktu Query -> CSV: 4.19 dtk | Parquet: 2.70 dtk


[Stage 68:>                                                         (0 + 2) / 2]

,Metrik,Sebelum,Sesudah
0,Total Baris,2010000.000,1987007
1,Order Unik,2000000.000,1987007
2,Duplikat,10000.000,0
3,Null Qty (%),0.299,0
4,Null Harga (%),0.300,0
5,Harga Negatif,2005.000,0
6,Outlier Qty (>50),1035.000,0
7,Variasi Nama Kota,24.000,8


In [12]:
spark.stop()